# Lab: HIPAA-Compliant Patient Analytics

## Learning Objectives
By the end of this lab, you will be able to:
1. **Configure Unity Catalog** for healthcare data governance (with production architecture guidance)
2. **Implement Dynamic Data Masking** using column-level security to protect PHI
3. **Apply Row-Level Security** to restrict data access by department/region
4. **Classify sensitive data** using Unity Catalog tags
5. **Monitor data access** using system audit logs for HIPAA compliance reporting

## HIPAA Quick Reference
| HIPAA Requirement | Databricks Feature | This Lab |
|---|---|---|
| Access Controls (§164.312(a)) | Unity Catalog RBAC | ✅ Section 2-4 |
| Minimum Necessary (§164.502(b)) | Column Masking + Row Filters | ✅ Section 3-4 |
| Audit Controls (§164.312(b)) | System Audit Logs | ✅ Section 5 |
| Data Classification | UC Tags + Comments | ✅ Section 2 |
| Encryption at Rest | Managed by AWS/Databricks | 📝 Explained |
| Encryption in Transit | TLS 1.2+ (built-in) | 📝 Explained |

## Prerequisites & Recommended Run Order
1. ✅ **`00a - Setup Groups and Users`** — Creates security groups (`phi_authorized`, `research_analysts`, etc.) and lets you assign yourself to different groups to test masking interactively
2. ✅ **`00 - Setup Synthetic Patient Data`** — Generates 10K patients + clinical data into `hipaa_demo.clinical_data`
3. ➡️ **This notebook** — Main demo walkthrough

> **💡 Tip:** After running the setup notebooks, try adding yourself to `research_analysts` in notebook 00a, then come back here and run the masking demo cells to see partial masking in action!

---

---
# Section 1: Unity Catalog Setup & Healthcare Data Architecture

## What is Unity Catalog?
Unity Catalog is Databricks' **unified governance solution** that provides:
- Centralized access control across all data assets (tables, volumes, models, functions)
- Fine-grained permissions (column masking, row filters)
- Data lineage tracking
- Audit logging for compliance

## Three-Level Namespace
```
Catalog (hipaa_demo)
  └── Schema (clinical_data)
        ├── Table (patients)
        ├── Table (encounters)
        ├── Table (diagnoses)
        ├── Table (lab_results)
        └── Table (medications)
```

## Production Architecture: IAM Roles & S3 Storage

> **⚠️ Trial Workspace Limitation:** The steps below cannot be fully executed on a free trial. They are documented here so you understand what a **production HIPAA-compliant deployment** requires.

### 1. AWS IAM Configuration (Production)

In production, you need a dedicated IAM role for the Unity Catalog metastore:

```json
// IAM Trust Policy for Unity Catalog Metastore
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "AWS": "arn:aws:iam::DATABRICKS_ACCOUNT_ID:role/unity-catalog-prod"
      },
      "Action": "sts:AssumeRole",
      "Condition": {
        "StringEquals": {
          "sts:ExternalId": "YOUR_EXTERNAL_ID"
        }
      }
    }
  ]
}
```

```json
// S3 Bucket Policy for HIPAA Data
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject",
        "s3:PutObject",
        "s3:DeleteObject",
        "s3:ListBucket",
        "s3:GetBucketLocation"
      ],
      "Resource": [
        "arn:aws:s3:::hipaa-clinical-data-prod",
        "arn:aws:s3:::hipaa-clinical-data-prod/*"
      ]
    }
  ]
}
```

### 2. S3 Bucket Requirements for HIPAA
| Setting | Required Value | Why |
|---------|---------------|-----|
| Server-Side Encryption | AES-256 or AWS KMS | HIPAA §164.312(a)(2)(iv) |
| Versioning | Enabled | Data recovery & audit trail |
| Access Logging | Enabled → audit bucket | HIPAA §164.312(b) |
| Public Access | **Block ALL** | Prevent data exposure |
| VPC Endpoint | Recommended | Keep traffic on private network |
| Cross-Region Replication | Optional | Disaster recovery |

### 3. Storage Credential & External Location (Production SQL)
```sql
-- These commands require account admin privileges and a configured IAM role
-- CREATE STORAGE CREDENTIAL hipaa_storage_cred
-- WITH (AWS_IAM_ROLE = 'arn:aws:iam::123456789012:role/unity-catalog-hipaa');

-- CREATE EXTERNAL LOCATION hipaa_clinical_storage
-- URL 's3://hipaa-clinical-data-prod/clinical/'
-- WITH (STORAGE CREDENTIAL hipaa_storage_cred)
-- COMMENT 'HIPAA-compliant storage for clinical data';
```

### 4. What We CAN Do on This Trial Workspace
The trial workspace comes with:
- ✅ **Unity Catalog** pre-configured with a default metastore
- ✅ **Managed storage** (Databricks-managed S3 bucket with encryption)
- ✅ **Catalog creation** privileges
- ✅ **Column masking & row filters**
- ✅ **Tags and data classification**
- ✅ **Audit logs** (system tables)

Let's verify our setup works ⬇️

## ⚠️ Before You Continue: Create an Account-Level Group

The next cell transfers table ownership to the **`account_managers`** group. This must be an **account-level** group (not a workspace-local group) for Unity Catalog ownership to work.

### Steps to Create the Group

1. Go to the [Databricks Account Console](https://accounts.cloud.databricks.com) and log in as an **account admin**.
2. In the left sidebar, click **User management**.
3. Select the **Groups** tab.
4. Click **Add group**.
5. Enter the group name: **`account_managers`** (must match exactly).
6. Click **Save**.
7. In the sidebar, click Workspaces.
8. Click your workspace name (workspace).
9. On the Permissions tab, click Add permissions.
10. Search for and select the group **account_manager**, assign the permission level (Admin), and then click Save
11. Return to this notebook and run the next cell.

### Important Notes

* **Workspace-local groups will not work** — Unity Catalog securables (catalogs, schemas, tables) can only be owned by account-level groups or individual users.
* If you see the error `PRINCIPAL_DOES_NOT_EXIST`, double-check the group name spelling and confirm it was created at the **account level**, not the workspace level.
* The group must be synced to this workspace. Account-level groups are automatically available in all workspaces that are assigned to your account's metastore.

In [0]:
%run "./00a - Setup Groups and Users"

In [0]:
%run "./00 - Setup Synthetic Patient Data"

In [0]:
%sql
-- =============================================================================
-- Verify our catalog and schema are set up correctly
-- =============================================================================

USE CATALOG hipaa_demo;
USE SCHEMA clinical_data;

-- Show all tables and their row counts
SELECT 
  'patients' AS table_name, COUNT(*) AS row_count FROM patients
UNION ALL
SELECT 'encounters', COUNT(*) FROM encounters
UNION ALL
SELECT 'diagnoses', COUNT(*) FROM diagnoses
UNION ALL
SELECT 'lab_results', COUNT(*) FROM lab_results
UNION ALL
SELECT 'medications', COUNT(*) FROM medications
ORDER BY table_name

In [0]:
%sql
-- =============================================================================
-- Look at the patient data BEFORE we apply any masking
-- Notice: ALL PHI fields are fully visible (SSN, name, DOB, address, etc.)
-- This is what we need to PROTECT!
-- =============================================================================

SELECT 
  patient_id,
  first_name,
  last_name,
  ssn,
  date_of_birth,
  phone_number,
  email,
  address,
  city,
  state,
  insurance_type,
  region,
  primary_department
FROM hipaa_demo.clinical_data.patients
LIMIT 10

---
# Section 2: Data Classification with Unity Catalog Tags

## Why Classify Healthcare Data?
HIPAA requires organizations to identify and categorize Protected Health Information (PHI). Unity Catalog **tags** let you:
- Label columns containing PHI, PII, or sensitive data
- Create policies based on classification
- Generate compliance reports showing where sensitive data lives
- Enable automated governance workflows

## PHI Categories (HIPAA Safe Harbor, §164.514)
The 18 HIPAA identifiers that constitute PHI:
1. Names
2. Geographic data (smaller than state)
3. Dates (except year) related to an individual
4. Phone numbers
5. Fax numbers
6. Email addresses
7. Social Security numbers
8. Medical record numbers
9. Health plan beneficiary numbers
10. Account numbers
11. Certificate/license numbers
12. Vehicle identifiers
13. Device identifiers
14. Web URLs
15. IP addresses
16. Biometric identifiers
17. Full-face photos
18. Any other unique identifying characteristic

In [0]:
%sql
-- =============================================================================
-- Apply sensitivity tags to columns containing PHI
-- Tags enable automated governance and compliance reporting
-- =============================================================================

-- First, set table-level tags
ALTER TABLE hipaa_demo.clinical_data.patients 
  SET TAGS ('data_classification' = 'PHI', 
            'hipaa_covered' = 'true',
            'retention_policy' = '7_years',
            'data_owner' = 'clinical_ops');

-- Tag individual PHI columns
ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN first_name SET TAGS ('phi_category' = 'name', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN last_name SET TAGS ('phi_category' = 'name', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN ssn SET TAGS ('phi_category' = 'ssn', 'sensitivity' = 'critical');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN date_of_birth SET TAGS ('phi_category' = 'date', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN address SET TAGS ('phi_category' = 'geographic', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN phone_number SET TAGS ('phi_category' = 'phone', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN email SET TAGS ('phi_category' = 'email', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN insurance_id SET TAGS ('phi_category' = 'health_plan_id', 'sensitivity' = 'high');

-- Tag non-PHI columns as safe for analytics
ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN gender SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN race SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN insurance_type SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN blood_type SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

SELECT '✅ Tags applied successfully to patients table' AS status

In [0]:
%sql
-- =============================================================================
-- Tag remaining clinical tables
-- =============================================================================

-- Encounters table
ALTER TABLE hipaa_demo.clinical_data.encounters 
  SET TAGS ('data_classification' = 'clinical', 'hipaa_covered' = 'true');

-- Diagnoses table  
ALTER TABLE hipaa_demo.clinical_data.diagnoses 
  SET TAGS ('data_classification' = 'clinical', 'hipaa_covered' = 'true');

ALTER TABLE hipaa_demo.clinical_data.diagnoses 
  ALTER COLUMN icd10_code SET TAGS ('data_type' = 'medical_code', 'sensitivity' = 'medium');

-- Lab Results table
ALTER TABLE hipaa_demo.clinical_data.lab_results 
  SET TAGS ('data_classification' = 'clinical', 'hipaa_covered' = 'true');

ALTER TABLE hipaa_demo.clinical_data.lab_results 
  ALTER COLUMN result_value SET TAGS ('data_type' = 'clinical_measurement', 'sensitivity' = 'medium');

-- Medications table
ALTER TABLE hipaa_demo.clinical_data.medications 
  SET TAGS ('data_classification' = 'clinical', 'hipaa_covered' = 'true');

ALTER TABLE hipaa_demo.clinical_data.medications 
  ALTER COLUMN medication_name SET TAGS ('data_type' = 'prescription', 'sensitivity' = 'medium');

SELECT '✅ Tags applied to all clinical tables' AS status

In [0]:
%sql
-- =============================================================================
-- Generate a compliance report: Which columns contain PHI?
-- This query uses the information_schema to find all tagged columns
-- =============================================================================

SELECT 
  t.table_name,
  c.column_name,
  c.data_type,
  ct.tag_name,
  ct.tag_value
FROM hipaa_demo.information_schema.tables t
JOIN hipaa_demo.information_schema.columns c 
  ON t.table_schema = c.table_schema AND t.table_name = c.table_name
LEFT JOIN hipaa_demo.information_schema.column_tags ct
  ON c.table_schema = ct.schema_name 
  AND c.table_name = ct.table_name 
  AND c.column_name = ct.column_name
WHERE t.table_schema = 'clinical_data'
  AND ct.tag_name IS NOT NULL
ORDER BY t.table_name, c.ordinal_position, ct.tag_name

---
# Section 3: Dynamic Data Masking (Column-Level Security)

## How Column Masking Works in Unity Catalog

Column masking applies a **SQL function** to a column that transforms the data based on the querying user's identity or group membership.

```
User Query → Unity Catalog checks column mask → Applies masking function → Returns masked/unmasked data
```

### Key Concepts:
- **Masking Function**: A SQL UDF that returns masked or original values based on `IS_MEMBER()` checks
- **`IS_MEMBER('group_name')`**: Returns TRUE if the current user belongs to the specified group
- **`CURRENT_USER()`**: Returns the email of the currently authenticated user
- **Transparent**: Masking is automatic — users don't need to do anything different in their queries
- **Performant**: Applied at query time, no data duplication needed

### Our Access Model
| Group | Access Level | Use Case |
|-------|-------------|----------|
| `phi_authorized` | Full PHI access | Clinical staff, treating physicians |
| `research_analysts` | Partial masking (last 4 of SSN, year of birth) | Research team |
| `billing_team` | Name + insurance only | Revenue cycle |
| Default (no group) | Fully masked | Data scientists, general analysts |


In [0]:
%sql
-- =============================================================================
-- Create masking functions for each PHI column type
-- These functions check group membership to determine masking level
-- =============================================================================

USE CATALOG hipaa_demo;
USE SCHEMA clinical_data;

-- 1. SSN Masking: Full SSN for authorized, last 4 for research, fully masked for others
CREATE OR REPLACE FUNCTION mask_ssn(ssn STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN ssn
    WHEN IS_MEMBER('research_analysts') THEN CONCAT('***-**-', RIGHT(ssn, 4))
    ELSE '***-**-****'
  END;

In [0]:
%sql
-- 2. Name Masking: Full name for authorized, initials for research, redacted for others
CREATE OR REPLACE FUNCTION mask_first_name(first_name STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN first_name
    WHEN IS_MEMBER('research_analysts') THEN CONCAT(LEFT(first_name, 1), '***')
    WHEN IS_MEMBER('billing_team') THEN first_name
    ELSE '[REDACTED]'
  END;

CREATE OR REPLACE FUNCTION mask_last_name(last_name STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN last_name
    WHEN IS_MEMBER('research_analysts') THEN CONCAT(LEFT(last_name, 1), '***')
    WHEN IS_MEMBER('billing_team') THEN last_name
    ELSE '[REDACTED]'
  END;

In [0]:
%sql
-- 3. Date of Birth Masking: Full DOB for authorized, year only for research, redacted for others
CREATE OR REPLACE FUNCTION mask_dob(dob DATE)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN CAST(dob AS STRING)
    WHEN IS_MEMBER('research_analysts') THEN CONCAT(YEAR(dob), '-XX-XX')
    ELSE '[REDACTED]'
  END;

-- 4. Phone Number Masking
CREATE OR REPLACE FUNCTION mask_phone(phone STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN phone
    WHEN IS_MEMBER('billing_team') THEN phone
    ELSE '(XXX) XXX-XXXX'
  END;

-- 5. Email Masking
CREATE OR REPLACE FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@', SPLIT(email, '@')[1])
  END;

-- 6. Address Masking
CREATE OR REPLACE FUNCTION mask_address(addr STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN addr
    WHEN IS_MEMBER('billing_team') THEN addr
    ELSE '[REDACTED]'
  END;

-- 7. Insurance ID Masking
CREATE OR REPLACE FUNCTION mask_insurance_id(ins_id STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN ins_id
    WHEN IS_MEMBER('billing_team') THEN ins_id
    ELSE CONCAT('INS-*****', RIGHT(ins_id, 3))
  END;

SELECT '✅ All masking functions created successfully' AS status

In [0]:
%sql
-- =============================================================================
-- Apply the masking functions to the patients table columns
-- Once applied, ALL queries against these columns go through the mask
-- =============================================================================

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN ssn SET MASK mask_ssn;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN first_name SET MASK mask_first_name;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN last_name SET MASK mask_last_name;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN date_of_birth SET MASK mask_dob;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN phone_number SET MASK mask_phone;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN email SET MASK mask_email;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN address SET MASK mask_address;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN insurance_id SET MASK mask_insurance_id;

SELECT '✅ Column masks applied to 8 PHI columns on patients table' AS status

In [0]:
%sql
-- =============================================================================
-- NOW query the same patients table - PHI columns are automatically masked!
-- Compare this with the "Before Masking" query above (Cell 5)
-- 
-- Since we're likely NOT in any of the authorized groups on this trial,
-- you should see the most restrictive masking applied.
-- =============================================================================

SELECT 
  patient_id,
  first_name    AS first_name_masked,
  last_name     AS last_name_masked,
  ssn           AS ssn_masked,
  date_of_birth AS dob_masked,
  phone_number  AS phone_masked,
  email         AS email_masked,
  address       AS address_masked,
  insurance_id  AS insurance_id_masked,
  -- These columns are NOT masked (analytics-safe)
  gender,
  race,
  insurance_type,
  blood_type,
  region,
  primary_department
FROM hipaa_demo.clinical_data.patients
LIMIT 10

## 🔍 Understanding the Masking Results

Compare the two queries above ("Before Masking" vs "With Masking Active"):

| Column | Before | After (Default/No Group) | After (research_analysts) | After (phi_authorized) |
|--------|--------|--------------------------|--------------------------|------------------------|
| `first_name` | Mary | [REDACTED] | M*** | Mary |
| `last_name` | Smith | [REDACTED] | S*** | Smith |
| `ssn` | 123-45-6789 | ***-**-**** | ***-**-6789 | 123-45-6789 |
| `date_of_birth` | 1985-03-15 | [REDACTED] | 1985-XX-XX | 1985-03-15 |
| `phone_number` | (555) 123-4567 | (XXX) XXX-XXXX | (XXX) XXX-XXXX | (555) 123-4567 |
| `email` | mary.smith@gmail.com | ma***@gmail.com | ma***@gmail.com | mary.smith@gmail.com |
| `address` | 123 Main St | [REDACTED] | [REDACTED] | 123 Main St |
| `insurance_id` | INS-123456789 | INS-*****789 | INS-*****789 | INS-123456789 |

### Key Takeaway
The **same SQL query** returns different results based on WHO runs it — this is the power of Unity Catalog column masking. No views, no application logic, no code changes needed.

### The Owner Bypass
**Important:** Table owners and catalog owners can see unmasked data. In production, use service principals as table owners rather than individual users to prevent bypass.

In [0]:
%sql
-- =============================================================================
-- Check which columns have masks applied (metadata inspection)
-- =============================================================================

SELECT
  c.table_name,
  c.column_name,
  c.data_type,
  CASE 
    WHEN c.column_name IN ('ssn','first_name','last_name','date_of_birth',
                           'phone_number','email','address','insurance_id')
    THEN '✅ MASKED'
    ELSE '❌ Not masked'
  END AS mask_status
FROM hipaa_demo.information_schema.columns c
WHERE c.table_schema = 'clinical_data'
  AND c.table_name = 'patients'
ORDER BY c.ordinal_position

---
# Section 4: Row-Level Security (Row Filters)

## How Row Filters Work
Row filters restrict **which rows** a user can see based on their identity or group membership. This implements the HIPAA **Minimum Necessary** principle — users only see data relevant to their role.

### Use Cases in Healthcare
| Scenario | Filter Logic |
|----------|-------------|
| Regional access | Cardiologist in Northeast only sees Northeast patients |
| Department access | Oncology staff only sees oncology encounters |
| Research cohorts | Researchers only see patients who consented to research |
| Multi-tenant | Hospital A only sees Hospital A data in a shared platform |

### How It Works
```
1. User runs: SELECT * FROM encounters
2. Unity Catalog checks: Does this table have a row filter?
3. Row filter function evaluates for EACH row:
   - Is user in 'all_regions' group? → Return all rows
   - Is user in 'northeast_team'? → Only return rows where region = 'Northeast'
   - Default? → Return only their assigned region
4. User only sees their authorized rows (transparently!)
```

> **📝 Trial Note:** Similar to column masks, `IS_MEMBER()` will return FALSE for groups that don't exist. We'll create the filters and demonstrate the behavior, plus show how to use `CURRENT_USER()` for user-level filtering.

In [0]:
%sql
-- =============================================================================
-- Create a row filter function for the encounters table
-- Restricts which rows users can see based on their region assignment
-- =============================================================================

USE CATALOG hipaa_demo;
USE SCHEMA clinical_data;

-- Row filter for encounters: Filter by region based on group membership
CREATE OR REPLACE FUNCTION encounters_region_filter(region_val STRING)
RETURNS BOOLEAN
RETURN 
  CASE
    -- Admins and compliance officers see everything
    WHEN IS_MEMBER('phi_authorized') THEN TRUE
    WHEN IS_MEMBER('compliance_officers') THEN TRUE
    -- Regional teams only see their region
    WHEN IS_MEMBER('northeast_team') AND region_val = 'Northeast' THEN TRUE
    WHEN IS_MEMBER('southeast_team') AND region_val = 'Southeast' THEN TRUE
    WHEN IS_MEMBER('midwest_team') AND region_val = 'Midwest' THEN TRUE
    WHEN IS_MEMBER('southwest_team') AND region_val = 'Southwest' THEN TRUE
    WHEN IS_MEMBER('west_team') AND region_val = 'West' THEN TRUE
    ELSE FALSE
  END;

SELECT '✅ Region filter function created' AS status

In [0]:
%sql
-- =============================================================================
-- Row filter for department-based access
-- In production, this ensures cardiologists only see cardiology data, etc.
-- =============================================================================

CREATE OR REPLACE FUNCTION encounters_dept_filter(dept_val STRING)
RETURNS BOOLEAN
RETURN 
  CASE
    WHEN IS_MEMBER('phi_authorized') THEN TRUE
    WHEN IS_MEMBER('compliance_officers') THEN TRUE
    WHEN IS_MEMBER('dept_cardiology') AND dept_val = 'Cardiology' THEN TRUE
    WHEN IS_MEMBER('dept_oncology') AND dept_val = 'Oncology' THEN TRUE
    WHEN IS_MEMBER('dept_neurology') AND dept_val = 'Neurology' THEN TRUE
    WHEN IS_MEMBER('dept_emergency') AND dept_val = 'Emergency' THEN TRUE
    WHEN IS_MEMBER('dept_surgery') AND dept_val = 'Surgery' THEN TRUE
    WHEN IS_MEMBER('dept_pediatrics') AND dept_val = 'Pediatrics' THEN TRUE
    WHEN IS_MEMBER('dept_orthopedics') AND dept_val = 'Orthopedics' THEN TRUE
    WHEN IS_MEMBER('dept_internal_med') AND dept_val = 'Internal Medicine' THEN TRUE
    ELSE TRUE  -- In production, change to FALSE for default-deny
  END;

SELECT '✅ Department filter function created' AS status

In [0]:
%sql
-- =============================================================================
-- Apply the row filter to the encounters table
-- Note: A table can only have ONE row filter at a time.
-- We'll use the region filter as the primary demonstration.
-- =============================================================================

-- Apply region-based row filter to encounters
ALTER TABLE hipaa_demo.clinical_data.encounters 
  SET ROW FILTER encounters_region_filter ON (region);

SELECT '✅ Row filter applied to encounters table on region column' AS status

In [0]:
%sql
-- =============================================================================
-- Query encounters with the row filter active
-- As the table owner, you'll see all rows.
-- In production, non-owner users would only see their region's data.
-- =============================================================================

-- Show row counts by region (table owner sees all)
SELECT 
  region,
  COUNT(*) AS encounter_count,
  COUNT(DISTINCT patient_id) AS unique_patients,
  ROUND(AVG(total_charges), 2) AS avg_charges
FROM hipaa_demo.clinical_data.encounters
GROUP BY region
ORDER BY region

In [0]:
%sql
-- =============================================================================
-- PRODUCTION EXAMPLE: Default-deny row filter
-- This is what you'd use in production (uncomment the ALTER TABLE to apply)
-- 
-- This function denies access to ANY row unless the user is in an 
-- authorized group. Much more secure than the demo version above.
-- =============================================================================

CREATE OR REPLACE FUNCTION encounters_region_filter_prod(region_val STRING)
RETURNS BOOLEAN
RETURN 
  CASE
    -- Admins see everything
    WHEN IS_MEMBER('phi_authorized') THEN TRUE
    WHEN IS_MEMBER('compliance_officers') THEN TRUE
    -- Regional teams see only their region
    WHEN IS_MEMBER('northeast_team') AND region_val = 'Northeast' THEN TRUE
    WHEN IS_MEMBER('southeast_team') AND region_val = 'Southeast' THEN TRUE  
    WHEN IS_MEMBER('midwest_team') AND region_val = 'Midwest' THEN TRUE
    WHEN IS_MEMBER('southwest_team') AND region_val = 'Southwest' THEN TRUE
    WHEN IS_MEMBER('west_team') AND region_val = 'West' THEN TRUE
    -- DEFAULT DENY: If user is not in any group, they see NOTHING
    ELSE FALSE
  END;

-- To apply in production (uncomment):
-- ALTER TABLE hipaa_demo.clinical_data.encounters 
--   SET ROW FILTER encounters_region_filter_prod ON (region);

SELECT '✅ Production-grade row filter created (not applied - see comments)' AS status

In [0]:
%sql
-- =============================================================================
-- Create pre-built analytics views that combine masking + row filters
-- These provide safe, de-identified datasets for different user groups
-- =============================================================================

-- View 1: De-identified patient cohort for research
CREATE OR REPLACE VIEW hipaa_demo.clinical_data.v_research_cohort AS
SELECT 
  -- De-identified: use hash instead of patient_id
  SHA2(patient_id, 256) AS research_subject_id,
  -- Demographics (no direct identifiers)
  gender,
  race,
  ethnicity,
  YEAR(date_of_birth) AS birth_year,
  FLOOR(DATEDIFF(CURRENT_DATE(), date_of_birth) / 365.25) AS age,
  -- Generalize geography to state level (HIPAA Safe Harbor)
  state,
  region,
  -- Clinical attributes
  insurance_type,
  blood_type,
  primary_department,
  is_active
FROM hipaa_demo.clinical_data.patients;

-- View 2: Population health summary (fully aggregated, no PHI)
CREATE OR REPLACE VIEW hipaa_demo.clinical_data.v_population_health AS
SELECT
  p.region,
  p.primary_department,
  p.gender,
  p.insurance_type,
  COUNT(DISTINCT p.patient_id) AS patient_count,
  COUNT(DISTINCT e.encounter_id) AS encounter_count,
  ROUND(AVG(e.total_charges), 2) AS avg_charges,
  ROUND(AVG(e.length_of_stay), 1) AS avg_length_of_stay,
  COUNT(DISTINCT CASE WHEN e.encounter_type = 'Emergency' THEN e.encounter_id END) AS er_visits,
  COUNT(DISTINCT d.diagnosis_id) AS total_diagnoses
FROM hipaa_demo.clinical_data.patients p
LEFT JOIN hipaa_demo.clinical_data.encounters e ON p.patient_id = e.patient_id
LEFT JOIN hipaa_demo.clinical_data.diagnoses d ON e.encounter_id = d.encounter_id
GROUP BY p.region, p.primary_department, p.gender, p.insurance_type;

-- View 3: Top diagnoses by region (de-identified)
CREATE OR REPLACE VIEW hipaa_demo.clinical_data.v_top_diagnoses AS
SELECT
  e.region,
  d.icd10_code,
  d.diagnosis_description,
  COUNT(*) AS diagnosis_count,
  COUNT(DISTINCT d.patient_id) AS affected_patients,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY e.region), 2) AS pct_of_region
FROM hipaa_demo.clinical_data.diagnoses d
JOIN hipaa_demo.clinical_data.encounters e ON d.encounter_id = e.encounter_id
GROUP BY e.region, d.icd10_code, d.diagnosis_description;

SELECT '✅ Three analytics views created' AS status

In [0]:
%sql
-- =============================================================================
-- Example: Safe analytics query using de-identified view
-- Data scientists can run this without any PHI exposure!
-- =============================================================================

-- Top 10 diagnoses across all regions
SELECT 
  icd10_code,
  diagnosis_description,
  SUM(diagnosis_count) AS total_cases,
  SUM(affected_patients) AS total_patients_affected
FROM hipaa_demo.clinical_data.v_top_diagnoses
GROUP BY icd10_code, diagnosis_description
ORDER BY total_cases DESC
LIMIT 10

---
# Section 5: Audit Logging & HIPAA Compliance Monitoring

## HIPAA Audit Requirements (§164.312(b))
HIPAA requires covered entities to:
1. **Record** all access to systems containing ePHI
2. **Review** activity logs regularly
3. **Investigate** anomalous access patterns
4. **Retain** audit logs for at least 6 years

## Databricks System Tables for Audit
Databricks provides **system tables** in the `system` catalog that capture comprehensive audit data:

| System Table | What It Captures |
|---|---|
| `system.access.audit` | All API calls, data access, authentication events |
| `system.information_schema.*` | Metadata about catalogs, schemas, tables, columns |
| `system.billing.usage` | Compute usage (useful for cost allocation per department) |

### Key Audit Events for HIPAA
| Event | Action Name | Why It Matters |
|-------|------------|----------------|
| Table query | `commandSubmit` | Who accessed patient data? |
| Permission change | `updatePermissions` | Was access level changed? |
| Data export | `downloadResults` | Was PHI downloaded? |
| Login | `login` | Authentication tracking |
| Table modification | `alterTable` | Was schema/security changed? |

> **⚠️ Trial Workspace Note:** System audit tables (`system.access.audit`) may require enablement. If the queries below fail, we provide alternative approaches using information_schema and manual audit views.

---

In [0]:
%sql
-- =============================================================================
-- Check if system audit tables are available on this workspace
-- =============================================================================

-- Try to access the audit log (may not be enabled on all trial workspaces)
SELECT 
  'system.access.audit' AS table_name,
  CASE 
    WHEN COUNT(*) >= 0 THEN '✅ Available'
    ELSE '❌ Not available'
  END AS status
FROM system.access.audit
WHERE event_date >= CURRENT_DATE() - INTERVAL 1 DAY
LIMIT 1

In [0]:
%sql
-- =============================================================================
-- Query 1: Who accessed our clinical data tables recently?
-- This query shows all data access events for our HIPAA-covered tables
-- =============================================================================

SELECT 
  event_date,
  event_time,
  user_identity.email AS user_email,
  action_name,
  request_params.full_name_arg AS table_accessed,
  source_ip_address,
  user_agent,
  response.status_code AS status
FROM system.access.audit
WHERE event_date >= CURRENT_DATE() - INTERVAL 7 DAY
  AND (
    request_params.full_name_arg LIKE 'hipaa_demo.clinical_data.%'
    OR request_params.full_name_arg LIKE 'hipaa_demo.%'
  )
ORDER BY event_time DESC
LIMIT 50

In [0]:
%sql
-- =============================================================================
-- Query 2: Access frequency heatmap - which users access which tables most?
-- Critical for identifying unusual access patterns (HIPAA requirement)
-- =============================================================================

SELECT 
  user_identity.email AS user_email,
  request_params.full_name_arg AS table_name,
  COUNT(*) AS access_count,
  MIN(event_time) AS first_access,
  MAX(event_time) AS last_access,
  COUNT(DISTINCT event_date) AS days_accessed
FROM system.access.audit
WHERE event_date >= CURRENT_DATE() - INTERVAL 30 DAY
  AND request_params.full_name_arg LIKE 'hipaa_demo.clinical_data.%'
  AND action_name IN ('commandSubmit', 'getTable', 'generateTemporaryTableCredential')
GROUP BY user_identity.email, request_params.full_name_arg
ORDER BY access_count DESC
LIMIT 25

In [0]:
%sql
-- =============================================================================
-- Query 3: Track all permission changes on clinical data
-- Any grant/revoke on HIPAA tables should be reviewed
-- =============================================================================

SELECT
  event_date,
  event_time,
  user_identity.email AS changed_by,
  action_name,
  request_params.full_name_arg AS affected_object,
  request_params.changes AS permission_changes,
  response.status_code AS status
FROM system.access.audit
WHERE event_date >= CURRENT_DATE() - INTERVAL 30 DAY
  AND action_name IN ('updatePermissions', 'setPermissions', 'grantPermission', 'revokePermission')
  AND (
    request_params.full_name_arg LIKE '%hipaa_demo%'
    OR request_params.full_name_arg LIKE '%clinical_data%'
  )
ORDER BY event_time DESC
LIMIT 25

In [0]:
%sql
-- =============================================================================
-- Create a reusable compliance summary view
-- This can be used in dashboards for ongoing HIPAA compliance monitoring
-- =============================================================================

CREATE OR REPLACE VIEW hipaa_demo.clinical_data.v_compliance_daily_summary AS
SELECT
  event_date,
  user_identity.email AS user_email,
  action_name,
  request_params.full_name_arg AS object_accessed,
  COUNT(*) AS event_count,
  MIN(event_time) AS first_event,
  MAX(event_time) AS last_event
FROM system.access.audit
WHERE request_params.full_name_arg LIKE 'hipaa_demo.%'
GROUP BY event_date, user_identity.email, action_name, request_params.full_name_arg;

SELECT * FROM hipaa_demo.clinical_data.v_compliance_daily_summary

---
# Section 6: Role-Based Access Control (RBAC) with Unity Catalog

## Implementing Least-Privilege Access

Unity Catalog uses a hierarchical permission model. For HIPAA compliance, implement the **principle of least privilege**:

### Permission Hierarchy
```
Catalog Owner (admin)
  └─ USAGE on catalog → Can see the catalog exists
       └─ USAGE on schema → Can see the schema exists
            ├─ SELECT on table → Can read data (subject to masks/filters)
            ├─ MODIFY on table → Can insert/update/delete data
            └─ EXECUTE on function → Can call functions
```

### Recommended HIPAA Role Definitions
| Role | Catalog | Schema | Tables | Functions | Use Case |
|------|---------|--------|--------|-----------|----------|
| **Compliance Officer** | USAGE | USAGE | SELECT (all) | EXECUTE | Audit & review |
| **Clinical Staff** | USAGE | USAGE | SELECT (with full PHI) | - | Direct patient care |
| **Research Analyst** | USAGE | USAGE | SELECT (masked PHI) | - | IRB-approved research |
| **Billing Staff** | USAGE | USAGE | SELECT (name + insurance) | - | Revenue cycle |
| **Data Scientist** | USAGE | USAGE | SELECT (views only) | - | Population analytics |
| **ML Engineer** | USAGE | USAGE | SELECT (de-identified) | EXECUTE | Model training |

### Production SQL for Permission Grants
```sql
-- These require account-level groups to exist first

-- Compliance officers: full read access + audit logs
-- GRANT USAGE ON CATALOG hipaa_demo TO compliance_officers;
-- GRANT USAGE ON SCHEMA hipaa_demo.clinical_data TO compliance_officers;
-- GRANT SELECT ON SCHEMA hipaa_demo.clinical_data TO compliance_officers;

-- Research analysts: read access (column masks auto-apply)
-- GRANT USAGE ON CATALOG hipaa_demo TO research_analysts;
-- GRANT USAGE ON SCHEMA hipaa_demo.clinical_data TO research_analysts;
-- GRANT SELECT ON SCHEMA hipaa_demo.clinical_data TO research_analysts;

-- Data scientists: only de-identified views
-- GRANT USAGE ON CATALOG hipaa_demo TO data_scientists;
-- GRANT USAGE ON SCHEMA hipaa_demo.clinical_data TO data_scientists;
-- GRANT SELECT ON TABLE hipaa_demo.clinical_data.v_research_cohort TO data_scientists;
-- GRANT SELECT ON TABLE hipaa_demo.clinical_data.v_population_health TO data_scientists;
-- GRANT SELECT ON TABLE hipaa_demo.clinical_data.v_top_diagnoses TO data_scientists;
-- (No access to base tables!)
```

In [0]:
%sql
-- =============================================================================
-- Show grants on the patients table (the most sensitive table)
-- In production, this should be tightly controlled
-- =============================================================================

SHOW GRANTS ON TABLE hipaa_demo.clinical_data.patients